In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [3]:
df=pd.read_csv("Social_Network_Ads.csv")
df.head(4)

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0


In [4]:
df=df.drop("User ID",axis=1)

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,confusion_matrix

In [6]:
df.info()
# no null values and 1 object column
# target = purchased

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Gender           400 non-null    object
 1   Age              400 non-null    int64 
 2   EstimatedSalary  400 non-null    int64 
 3   Purchased        400 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 12.6+ KB


In [7]:
cat_cols=["Gender","Purchased"]
num_cols=["Age","EstimatedSalary"]

In [10]:
for i in cat_cols:
    print(f"{i} : {df[i].value_counts()}")
# imbalanced dataset , Not purchased > Purchased

Gender : Gender
Female    204
Male      196
Name: count, dtype: int64
Purchased : Purchased
0    257
1    143
Name: count, dtype: int64


In [11]:
# there is no missing data, in case if missing data is there , so to replace the 
# null value - data imputation is done. In the pipeline it is done via SimpleImputer
num_pipeline=Pipeline(
    [
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ]
)

In [13]:
cat_pipeline=Pipeline(
    [
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [14]:
preprocessor=ColumnTransformer(
    [
        ("num",num_pipeline,num_cols),
        ("cat",cat_pipeline,["Gender"]) # purchased is target column so not putting it
    ]
)

In [15]:
features=df.drop("Purchased",axis=1)
target=df["Purchased"]

In [16]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest=train_test_split(features,target,random_state=0,
                                          test_size=0.2,stratify=target)
print(xtrain.shape,ytrain.shape)
print(xtest.shape,ytest.shape)

(320, 3) (320,)
(80, 3) (80,)


In [22]:
from sklearn.metrics import accuracy_score
best_acc=0
d={}
for k in range(1,51,2):
    pipeline=Pipeline(
        [
            ("preprocessor",preprocessor),
            ("knn",KNeighborsClassifier(n_neighbors=k))
        ]
     )
    pipeline.fit(xtrain,ytrain)
    ypred=pipeline.predict(xtest)
    acc=accuracy_score(ytest,ypred)
    d[k]=acc

In [23]:
d

{1: 0.8875,
 3: 0.9,
 5: 0.9125,
 7: 0.9125,
 9: 0.925,
 11: 0.925,
 13: 0.925,
 15: 0.925,
 17: 0.925,
 19: 0.925,
 21: 0.9125,
 23: 0.925,
 25: 0.8875,
 27: 0.8875,
 29: 0.8625,
 31: 0.8625,
 33: 0.875,
 35: 0.875,
 37: 0.85,
 39: 0.825,
 41: 0.825,
 43: 0.825,
 45: 0.8375,
 47: 0.85,
 49: 0.85}

In [24]:
# key with maximum value
max_key=max(d,key=d.get)
max_key

9

In [25]:
model=Pipeline(
        [
            ("preprocessor",preprocessor),
            ("knn",KNeighborsClassifier(n_neighbors=9))
        ]
     )
model.fit(xtrain,ytrain)
ypred=model.predict(xtest)
acc=accuracy_score(ytest,ypred)
print(f"Training Score : {model.score(xtrain,ytrain)}")
print(f"Testing Score : {model.score(xtest,ytest)}")
print(pd.DataFrame(confusion_matrix(ytest,ypred),
                   columns=["Not Purchased","Purchased"],
                   index=["Not Purchased","Purchased"])
     )
print(classification_report(ytest,ypred))

Training Score : 0.90625
Testing Score : 0.925
               Not Purchased  Purchased
Not Purchased             48          3
Purchased                  3         26
              precision    recall  f1-score   support

           0       0.94      0.94      0.94        51
           1       0.90      0.90      0.90        29

    accuracy                           0.93        80
   macro avg       0.92      0.92      0.92        80
weighted avg       0.93      0.93      0.93        80



In [26]:
a=[int(input("Enter number : "))for i in range(5)]
print(a)

Enter number :  1
Enter number :  22
Enter number :  3
Enter number :  4
Enter number :  5


[1, 22, 3, 4, 5]
